# Lecture 5 — Agentic AI Systems
## Agno Agent: Web Search + RAG + TAG — Complete Class Notebook
### AI Learning Academy | Applied AI Institute 2026

This notebook walks through the **complete build of a Movie Recommendation Agent** used as the
running example throughout Lecture 5. Each section maps directly to a slide-deck section:

| Notebook Section 
|---|
| §0 — Setup & Imports 
| §1 — Model Initialization 
| §2 — Agent Functional Design (Role → Memory → Success Criteria → Metrics) 
| §3 — Knowledge, Skills & Tools 
| §4 — Orchestration & Pydantic Models 
| §5 — Integration: REST API, MCP, A2A 
| §6 — Advanced Topics: Debug, Context 
| §7 — Multi-Agent Team Pattern 
| §8 — Full Interactive Chatbot 

---
## §0 — Setup & Imports

In [ ]:
# Install all required packages
!pip install agno openai python-dotenv pandas openpyxl requests chromadb pydantic --quiet

In [1]:
# ── Core Agno ─────────────────────────────────────────────────────────────────
from agno.agent import Agent
from agno.models.message import Message
from agno.db.sqlite import SqliteDb
from agno.tools.serper import SerperTools
from agno.tools import tool, Toolkit
from agno.models.azure import AzureOpenAI
from agno.vectordb.chroma import ChromaDb
from agno.knowledge.chunking.row import RowChunking
from agno.knowledge.embedder.azure_openai import AzureOpenAIEmbedder
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.reader.excel_reader import ExcelReader
from agno.knowledge.reader.pdf_reader import PDFReader

# ── Pydantic (§4 – structured output) ─────────────────────────────────────────
from pydantic import BaseModel, Field
from typing import List, Optional

# ── Standard library ──────────────────────────────────────────────────────────
import os, json, requests
import pandas as pd
from dotenv import load_dotenv
from os import getenv
from pathlib import Path

print('✅  All imports successful')


✅  All imports successful


In [2]:
# ── Load environment variables from .env file ─────────────────────────────────

load_dotenv()
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [64]:


# ── File paths ────────────────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # ← change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'

agent_db       = SqliteDb(db_file=MEMORY_DB)

# ── Constants ─────────────────────────────────────────────────────────────────
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL')
EMBED_MODEL    = 'text-embedding-3-small'
MAX_ABT_RESULTS = 5
RAG_TOP_K       = 4
SESSION_ID      = 'movie-bot-session-001'   # change per user

print(f'Data folder : {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder : /Users/asathi/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


---
## §1 — Model Initialization
**Agent V1** · How Agno Builds an Agent: Model → Instructions → Tools → Memory

In [4]:
# ── Agent V1 : How Agno Builds Agent ──────────────────────────────────────────
# Step ①  Model
model = AzureOpenAI(
    id             = getenv("LLM-MODEL"),
    api_version    = getenv("LLM_MODEL_VERSION"),
    azure_endpoint = getenv("AZURE_INFERENCE_ENDPOINT")
)

# Quick smoke-test
response = model.response(messages=[Message(role="user", content="Hello")])
print('Model response:', response.content)

Model response: Hello! How can I assist you today?


---
## §2 — Agent Functional Design
**Agent V1** · Progressively build the system prompt:
Role → Skills → Orchestration → Memory → Success Criteria → Performance Metrics

Each subsection adds one new element — exactly mirroring the slide-by-slide build.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V1 — Role and Skills
# ─────────────────────────────────────────────────────────────────────────────
ROLE_AND_SKILLS = """You are a friendly and knowledgeable movie expert.
Your ONLY role is to help users discover and learn about movies.
Do not answer questions unrelated to movies.
"""

# ─────────────────────────────────────────────────────────────────────────────
# Agent V1 with Role, Skills, Orchestration and Memory
# Orchestration (Flipped Interaction)
# ─────────────────────────────────────────────────────────────────────────────
ORCHESTRATION = """
UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief,
engaging conversation. Ask ONE leading question at a time across these
dimensions (do not ask all at once):
  • Viewing context : Are you watching alone, with family, friends, or a date?
  • Audience        : adults, teens, young children, mixed group?
  • Genre mood      : action, comedy, thriller, romance, drama, sci-fi, documentary?
  • Decade/era      : classic, 1980s–90s, modern, recent?
  • Country/language: any preference for country of origin or spoken language?
  • Oscar/award     : interested in critically acclaimed / award-nominated films?
Stop asking when you have enough to make a personalised recommendation
(typically 2-3 answers).
"""

# ─────────────────────────────────────────────────────────────────────────────
# Agent V1 with Memory
# ─────────────────────────────────────────────────────────────────────────────
MEMORY_INSTRUCTION = """
MEMORY
You have memory of user preferences (automatically provided in context). Use this to:
  - Tailor recommendations to their interests
  - Consider their preferences across turns
  - Reference who they are watching with
"""

# Build Agent V1 prompt (Role + Skills + Orchestration + Memory)
PROMPT_AGENTV1 = ROLE_AND_SKILLS + ORCHESTRATION + MEMORY_INSTRUCTION

# Minimal agent — matches v1 code example
agent_v1 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'agent_v1-demo',
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV1 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent V1 created (Role + Skills + Orchestration + Memory)')

✅  Agent V1 created (Role + Skills + Orchestration + Memory)


In [10]:
# Test Agent V1
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v1.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v1.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes Samurai fiction"
# Follow on query "How about a movie like the series Shogun"
# "More Ramance less Samurai"
# "English only"

Your input:  Exit
Goodbye!


In [14]:
chat_memory = agent_v1.get_chat_history()


print("\nSession Memory:")
for m in chat_memory:
    print("-", m.content)


print("\n" + "-"*50 + "\n")


Session Memory:
- Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction
- Great combination! For your date night, would you prefer a movie that leans more toward the romantic side with some samurai elements, or a samurai film that includes a strong romantic storyline?
- How about a movie like the series Shogun
- Perfect! Since you like the series *Shogun*—a historical samurai drama with rich romance and political intrigue—would you prefer a film set in a similar historical Japan setting, or are you open to samurai-themed movies with a bit more action or fantasy elements mixed in?
- More Romance  less Samurai
- Got it—more romance with samurai as a backdrop. Are you interested in a classic tale or something modern that blends romance and samurai themes? Also, do you prefer a Japanese-language film or an English-language one?
- English language
- Thanks for that! One English-language movie that beautifully combines romance with a samu

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V2 — Add SUCCESS CRITERIA
# ─────────────────────────────────────────────────────────────────────────────
SUCCESS_CRITERIA = """
SUCCESS CRITERIA
A successful recommendation interaction is one where:
  1. The agent asks at least 2 clarifying questions before recommending.
  2. Every recommendation includes: title, year, genre, synopsis (2–3 sentences),
     and a personalised explanation of why it matches the user's stated preferences.
  3. The agent cites a source (web link or knowledge base reference) for any
     factual claim about awards, ratings, or streaming availability.
  4. The user confirms or asks a follow-up — confirming the recommendation felt
     personalised rather than generic.
"""

PROMPT_AGENTV2 = PROMPT_AGENTV1 + SUCCESS_CRITERIA

agent_v2 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'agent_v2-demo',
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV2 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅   agent v2 created (added Success Criteria)')

✅   agent v2 created (added Success Criteria)


In [13]:
# Test Agent V2
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v2.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v2.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes Samurai fiction"
# Follow on query "How about a movie like the series Shogun"

Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction

Agent:
That sounds like a fun mix! To help me find the perfect movie, could you tell me if you prefer something more classic or modern for your date night? Also, would you like the movie to have more romance with samurai elements, or a balanced mix of both genres?

Session Memory:
- Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction
- That sounds like a fun mix! To help me find the perfect movie, could you tell me if you prefer something more classic or modern for your date night? Also, would you like the movie to have more romance with samurai elements, or a balanced mix of both genres?

--------------------------------------------------

Your input:  How about a movie like the series Shogun

Agent:
Great! Since you like the series Shogun, are you looking for a movie with historical samurai drama that also has 

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V3 — Add PERFORMANCE METRICS
# ─────────────────────────────────────────────────────────────────────────────
PERFORMANCE_METRICS = """
PERFORMANCE METRICS
Track the following per session to evaluate agent quality:
  • preference_signals_collected  : int   — target ≥ 2 before recommending
  • recommendation_accepted       : bool  — did the user engage positively?
  • source_citations_included     : int   — target ≥ 1 per factual claim
  • turns_to_first_recommendation : int   — target ≤ 4 turns
  • user_satisfaction_score       : 1–5   — collected via follow-up question
If turns_to_first_recommendation > 4, shorten your follow-up questions.
"""

PROMPT_AGENTV3 = PROMPT_AGENTV2 + PERFORMANCE_METRICS

agent_v3 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'Agent_v3-demo',
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV3 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent V3 created (added Performance Metrics)')
print()
print('─── Complete AGENT SYSTEM PROMPT ───')
print(PROMPT_AGENTV3, '...')

✅  Agent V3 created (added Performance Metrics)

─── Complete AGENT SYSTEM PROMPT ───
You are a friendly and knowledgeable movie expert.
Your ONLY role is to help users discover and learn about movies.
Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief,
engaging conversation. Ask ONE leading question at a time across these
dimensions (do not ask all at once):
  • Viewing context : Are you watching alone, with family, friends, or a date?
  • Audience        : adults, teens, young children, mixed group?
  • Genre mood      : action, comedy, thriller, romance, drama, sci-fi, documentary?
  • Decade/era      : classic, 1980s–90s, modern, recent?
  • Country/language: any preference for country of origin or spoken language?
  • Oscar/award     : interested in critically acclaimed / award-nominated films?
Stop asking when you have enough to make a personalised recommendation
(typicall

In [ ]:
# Test  Agent V3
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v3.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v3.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes Samurai fiction"
# Follow on query "How about a movie like the series Shogun"

Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction

Agent:
Great mix of interests for a date night! Just to make sure I find the perfect movie for you both, could you tell me if you prefer a movie that leans more toward romance with some samurai elements, or would you prefer a samurai film that has a romantic subplot? Also, do you have any preference for the movie's era—something classic, or something more modern?

Session Memory:
- Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction
- Great mix of interests for a date night! Just to make sure I find the perfect movie for you both, could you tell me if you prefer a movie that leans more toward romance with some samurai elements, or would you prefer a samurai film that has a romantic subplot? Also, do you have any preference for the movie's era—something classic, or something more modern?

-------------------------

## §3 — Knowledge,  Skills and Tools

---
** Agent V4 with Serper ** · 
** Agent V5 with TAG Knowledge
** Agent V6 with RAG Knowledge 

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V4 — Add Serper Tool
# ────────────────────────────────────────────────────────────────────────────
TOOL_INSTRUCTIONS = """We are providing you with Serper tool. Use this tool to search the web for your response. 
Do not use your LLM memory. Provide a reference to the source you found through Google search."""

PROMPT_AGENTV4 = PROMPT_AGENTV3 + TOOL_INSTRUCTIONS

agent_v4 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'Agent_v4-demo_3_22',
    tools                  = [SerperTools()],
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV4 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent V4 created (added Performance Metrics)')
print()
print('─── Complete AGENT SYSTEM PROMPT ───')
print(PROMPT_AGENTV4, '...')

✅  Agent V4 created (added Performance Metrics)

─── Complete AGENT SYSTEM PROMPT ───
You are a friendly and knowledgeable movie expert.
Your ONLY role is to help users discover and learn about movies.
Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief,
engaging conversation. Ask ONE leading question at a time across these
dimensions (do not ask all at once):
  • Viewing context : Are you watching alone, with family, friends, or a date?
  • Audience        : adults, teens, young children, mixed group?
  • Genre mood      : action, comedy, thriller, romance, drama, sci-fi, documentary?
  • Decade/era      : classic, 1980s–90s, modern, recent?
  • Country/language: any preference for country of origin or spoken language?
  • Oscar/award     : interested in critically acclaimed / award-nominated films?
Stop asking when you have enough to make a personalised recommendation
(typicall

In [20]:
# Test  Agent V4
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v4.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v4.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "I am interested in watching latest good movies.  
# "It is for a date night.  Which movie won 2026 Oscar for best picture?"


Your input:  I am interested in watching latest good movies.

Agent:
Great! Are you planning to watch alone or with someone? If with others, who will be watching with you (family, friends, date)?

Session Memory:
- I am interested in watching latest good movies.
- Great! Are you planning to watch alone or with someone? If with others, who will be watching with you (family, friends, date)?

--------------------------------------------------

Your input:  It is for a date night.  Which movie won 2026 Oscar for best picture?

Agent:
The movie that won the 2026 Oscar for Best Picture is "One Battle After Another." It also won awards for best director and best adapted screenplay. Are you interested in watching this Oscar-winning movie for your date night? Or would you like me to recommend other recent good movies that might suit the mood?

Session Memory:
- I am interested in watching latest good movies.
- Great! Are you planning to watch alone or with someone? If with others, who will be w

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# A Quick test to check Serper tool in action
# ────────────────────────────────────────────────────────────────────────────
TOOL_INSTRUCTIONS = """TOOL INSTRUCTIONS

We are providing you with Serper tool. Use this tool to search the web for your response. 
Do not use your LLM memory. Provide a reference to the source you found through Google search."""


agent_serper = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'Agent_serper-demo',
    tools                  = [SerperTools()],
    add_history_to_context = True,
    instructions           = TOOL_INSTRUCTIONS,
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent Serper created')
print()
print('─── Complete AGENT SYSTEM PROMPT ───')
print(TOOL_INSTRUCTIONS, '...')

✅  Agent Serper created

─── Complete AGENT SYSTEM PROMPT ───
TOOL INSTRUCTIONS

We are providing you with Serper tool. Use this tool to search the web for your response. 
Do not use your LLM memory. Provide a reference to the source you found through Google search. ...


In [16]:
# Test  Serper tool in Agent
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_serper.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_serper.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Which movie won 2026 Oscar for best picture?"

Your input:  Which movie won 2026 Oscar for best picture?

Agent:
The movie that won the 2026 Oscar for Best Picture is "One Battle After Another." 

Reference: [CNN - Oscars 2026 Best Picture Winner](https://www.cnn.com/entertainment/live-news/oscars-academy-awards-03-15-26)

Session Memory:
- Which movie won 2026 Oscar for best picture?
- None
- The movie that won the 2026 Oscar for Best Picture is "One Battle After Another." 

Reference: [CNN - Oscars 2026 Best Picture Winner](https://www.cnn.com/entertainment/live-news/oscars-academy-awards-03-15-26)

--------------------------------------------------

Your input:  Exit
Goodbye!


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# TAG Knowledge: Load Genre Taxonomy
# ─────────────────────────────────────────────────────────────────────────────
df_taxonomy   = pd.read_excel(TAXONOMY_FILE, sheet_name='custom_genre')
taxonomy_json = df_taxonomy.to_dict(orient='records')
taxonomy_json_string = json.dumps(taxonomy_json, indent=2)  # Convert to JSON string for prompt injection

print(f'Loaded {len(taxonomy_json)} custom genres')
print('Sample genres:', taxonomy_json_string[:500], '...')  # Print first 500 chars of JSON string for verification

Loaded 24 custom genres
Sample genres: [
  {
    "cluster_id": 5,
    "custom_genre": "Clever Heist Capers",
    "description": "This genre unites films centered around smart, stylish criminals and their intricate schemes, blending suspense, wit, and high-stakes cons or heists. These movies focus on the thrill of the plan, the dynamics within the crew, and the clever twists that keep audiences guessing.",
    "exemplar_overview": "When a seasoned thief assembles a diverse team to pull off the ultimate heist, they must outsmart rival  ...


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V5 — Add Custom Genre Taxonomy to Knowledge
# ────────────────────────────────────────────────────────────────────────────
TAXONOMY_INSTRUCTIONS = f"""In addition to standard genre definitions, 
you will use a custom genre list with specific genre definitions defined by your user
- Use the genre list to match genre to the user preferences
- Explain to the user the match with the genre
- Use the genre definition to find a movie that matches the genre{taxonomy_json_string}"""

PROMPT_AGENTV5 = PROMPT_AGENTV4 + TAXONOMY_INSTRUCTIONS

agent_v5 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'Agent_v5-demo',
    tools                  = [SerperTools()],
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV5 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent V5 created (added Custom Genre Taxonomy)')
print()
print('─── Complete AGENT SYSTEM PROMPT ───')
print(PROMPT_AGENTV5[:5000], '...')


✅  Agent V5 created (added Custom Genre Taxonomy)

─── Complete AGENT SYSTEM PROMPT ───
You are a friendly and knowledgeable movie expert.
Your ONLY role is to help users discover and learn about movies.
Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief,
engaging conversation. Ask ONE leading question at a time across these
dimensions (do not ask all at once):
  • Viewing context : Are you watching alone, with family, friends, or a date?
  • Audience        : adults, teens, young children, mixed group?
  • Genre mood      : action, comedy, thriller, romance, drama, sci-fi, documentary?
  • Decade/era      : classic, 1980s–90s, modern, recent?
  • Country/language: any preference for country of origin or spoken language?
  • Oscar/award     : interested in critically acclaimed / award-nominated films?
Stop asking when you have enough to make a personalised recommendation
(typica

In [56]:
# Test  Agent V5
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v5.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v5.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "I am interested in watching a movie for a date night.  
# "How about a global espionate thriller with a romantic subplot?"

Your input:  Exit
Goodbye!


In [58]:
chat_memory = agent_v5.get_chat_history()
print("\nSession Memory:")
for m in chat_memory:
   print("-", m.content)


Session Memory:
- I am interested in watching a movie for a date night. 
- That sounds great! To help me find the perfect movie for your date night, could you tell me if you prefer a particular genre or mood? For example, romantic, comedy, thriller, drama, or something else?
- How about a global espionate thriller with a romantic subplot?
- Global espionage thrillers with romantic subplots make for an exciting and captivating date night choice! To narrow it down further, do you have a preference for films from a certain era or decade, like classic spy films, modern ones, or recent releases?
- How about a new movie
- A recent global espionage thriller with a romantic subplot it is! Lastly, would you like the movie to be critically acclaimed, perhaps nominated for or winning awards, or are you open to any thrilling and entertaining film?
- Just a good entertaining movie
- Thank you for sharing your preferences! For a date night, I recommend the film:

**"Red Notice" (2021)**
- Genre: Gl

### Agent 6 — Build RAG Knowledge Base (ChromaDB + Excel ABT + Oscars PDF)

In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# RAG Knowledge: Build ChromaDB vector store
# ─────────────────────────────────────────────────────────────────────────────
embedder = AzureOpenAIEmbedder(
    id           = EMBED_MODEL,
    enable_batch = True,
    batch_size   = 10,
)


vector_db = ChromaDb(
    collection        = 'movie_abt',
    path              = 'tmp/chromadb',
    persistent_client = True,   # <-- persist data to disk
    embedder          = embedder
)


In [43]:
knowledge = Knowledge(vector_db=vector_db)
print('ChromaDB knowledge store initialised')

ChromaDB knowledge store initialised


In [62]:

# Access the internal ChromaDB client
collection = vector_db._client.get_collection("movie_abt")
results = collection.peek()
print(collection.count())

2013


In [65]:
# ─────────────────────────────────────────────────────────────────────────────
# Index Oscars PDF into the same knowledge base
# ─────────────────────────────────────────────────────────────────────────────
if OSCARS_PDF.exists():
    knowledge.insert(
        path           = OSCARS_PDF,
        reader         = PDFReader(),
        skip_if_exists = True,
    )
    print(f'✅  Oscars PDF indexed: {OSCARS_PDF.name}')
else:
    print('⚠️  Oscars PDF not found — skipping (place file at', OSCARS_PDF, ')')

INFO Adding content from path, eb822c93-d42f-522a-87ef-92dd90d4207a, None,                                         
     ../data/input/The_98th_Academy_Awards_2026.pdf, None

✅  Oscars PDF indexed: The_98th_Academy_Awards_2026.pdf


In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# Index Movie ABT in batches (row-level chunking — each row = one document)
# skip_if_exists=True means safe to re-run without re-embedding
# ─────────────────────────────────────────────────────────────────────────────
df_full_abt = pd.read_excel(ABT_FILE)
BATCH_SIZE  = 1000

batch_paths = []
for i, start in enumerate(range(0, len(df_full_abt), BATCH_SIZE)):
    bp = DATA_FOLDER / f'movie_abt_batch_{i+1}.xlsx'
    df_full_abt.iloc[start:start + BATCH_SIZE].to_excel(bp, index=False)
    batch_paths.append(bp)

print(f'Split ABT ({len(df_full_abt)} rows) into {len(batch_paths)} batches')



Split ABT (9742 rows) into 10 batches


In [53]:
# Index all batches
for bp in batch_paths[0:2]:
    knowledge.insert(
        path           = bp,
        reader         = ExcelReader(chunking_strategy=RowChunking()),
        skip_if_exists = True,
    )
    print(f'  Indexed: {bp.name}')

print('\n✅  ABT indexing complete')

INFO Adding content from path, 27856271-93cd-5956-9355-68f6d34a7040, None, ../data/input/movie_abt_batch_1.xlsx,   
     None

  Indexed: movie_abt_batch_1.xlsx


INFO Adding content from path, 2cd44313-356c-58e1-8278-a72c15604c75, None, ../data/input/movie_abt_batch_2.xlsx,   
     None

INFO Upserting 1001 documents

  Indexed: movie_abt_batch_2.xlsx

✅  ABT indexing complete


In [60]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent V6 — Use ABT and Oscar news using Knowledge
# ────────────────────────────────────────────────────────────────────────────
KNOWLEDGE_INSTRUCTIONS = f"""RETRIEVE FROM KNOWLEDGE BASE
  Search the RAG knowledge base for movies matching user preferences and recent Oscar news.
  Use retrieved knowledge to inform your recommendations, citing specific facts from the ABT and Oscars data to enhance the relevance and timeliness of your suggestions."""

PROMPT_AGENTV6 = KNOWLEDGE_INSTRUCTIONS 

agent_v6 = Agent(
    model                  = model,
    db                     = agent_db,
    session_id             = 'Agent_v6-demo_3_22',
    tools                  = [SerperTools()],
    knowledge              = knowledge,
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV6 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)
print('✅  Agent V6 created (added Knowledge integration)')
print()
print('─── Complete AGENT SYSTEM PROMPT ───')
print(PROMPT_AGENTV6[:5000], '...')

✅  Agent V6 created (added Knowledge integration)

─── Complete AGENT SYSTEM PROMPT ───
RETRIEVE FROM KNOWLEDGE BASE
  Search the RAG knowledge base for movies matching user preferences and recent Oscar news.
  Use retrieved knowledge to inform your recommendations, citing specific facts from the ABT and Oscars data to enhance the relevance and timeliness of your suggestions. ...


In [61]:
# Test  Agent V6
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v6.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v6.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "How many TMDB votes were there for the movie Toy Story?"
# "Which movies were nominated for best original screenplay for Oscars in 2026?"

Your input:  How many TMDB votes were there for the movie Toy Story?


INFO Found 10 documents


Agent:
The movie "Toy Story" has received 19,556 votes on TMDB (The Movie Database).

Session Memory:
- How many TMDB votes were there for the movie Toy Story?
- None
- The movie "Toy Story" has received 19,556 votes on TMDB (The Movie Database).

--------------------------------------------------

Your input:  Which movies were nominated for best original screenplay for Oscars in 2026?


INFO Found 10 documents


Agent:
The movies nominated for Best Original Screenplay at the 2026 Oscars are:

- "Bugonia"
- "Frankenstein"
- "Hamnet"
- "Marty Supreme"
- "One Battle After Another"
- "Sentimental Value"
- "Sinners"
- "Train Dreams"

If you would like more details about any of these films or their writers, feel free to ask!

Session Memory:
- How many TMDB votes were there for the movie Toy Story?
- None
- The movie "Toy Story" has received 19,556 votes on TMDB (The Movie Database).
- Which movies were nominated for best original screenplay for Oscars in 2026?
- None
- The movies nominated for Best Original Screenplay at the 2026 Oscars are:

- "Bugonia"
- "Frankenstein"
- "Hamnet"
- "Marty Supreme"
- "One Battle After Another"
- "Sentimental Value"
- "Sinners"
- "Train Dreams"

If you would like more details about any of these films or their writers, feel free to ask!

--------------------------------------------------

Your input:  Exit
Goodbye!


## §4 — Pydantic I/O, REST API, Orchestration

---
** Agent V7 with Pydantic I/O 

** Agent V8 with TMDB REST API

** Agent V9 with Orchesrtation


### V7 - Pydantic I/O

In [73]:
# ─────────────────────────────────────────────────────────────────────────────
# Pydantic Input Model (captures structured user preferences)
# ─────────────────────────────────────────────────────────────────────────────
class MoviePreference(BaseModel):
    """Structured user-preference object collected via Flipped Interaction."""
    viewing_context : str           = Field(..., description="alone | family | friends | date")
    audience        : str           = Field(..., description="adults | teens | children | mixed")
    genre_mood      : str           = Field(..., description="genre or emotional mood stated by user")
    era_preference  : Optional[str] = Field(None, description="classic | 1980s-90s | modern | recent")
    language_pref   : Optional[str] = Field(None, description="preferred spoken language")
    awards_interest : bool          = Field(False, description="prefers critically acclaimed films")

# ─────────────────────────────────────────────────────────────────────────────
# Pydantic Output Model (enforces recommendation structure)
# ─────────────────────────────────────────────────────────────────────────────
class MovieRecommendation(BaseModel):
    """A single validated movie recommendation."""
    title            : str           = Field(..., description="Movie title")
    year             : int           = Field(..., description="Release year (4 digits)")
    custom_genre     : str           = Field(..., description="Matched custom genre from taxonomy")
    movielens_genres : List[str]     = Field(..., description="Standard genre tags")
    synopsis         : str           = Field(..., description="2–3 sentence synopsis")
    why_match        : str           = Field(..., description="Why this matches the user's preferences")
    rating           : float         = Field(..., description="TMDB vote_average 0–10")
    streaming        : Optional[str] = Field(None, description="Streaming platform if known")
    source           : str           = Field(..., description="Source URL or knowledge-base reference")


class RecommendationResponse(BaseModel):
    """Top-level structured agent response."""
    recommendations    : List[MovieRecommendation] = Field(..., description="Ordered best-first list")
    follow_up_question : Optional[str]             = Field(None, description="Next clarifying question")


print('Pydantic models defined:')
print('  Input  →', list(MoviePreference.model_fields.keys()))
print('  Output →', list(MovieRecommendation.model_fields.keys()))

Pydantic models defined:
  Input  → ['viewing_context', 'audience', 'genre_mood', 'era_preference', 'language_pref', 'awards_interest']
  Output → ['title', 'year', 'custom_genre', 'movielens_genres', 'synopsis', 'why_match', 'rating', 'streaming', 'source']


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Agent with Pydantic structured output
# response_model enforces RecommendationResponse schema on every call
# ─────────────────────────────────────────────────────────────────────────────
STRUCTURED_OUTPUT_INSTRUCTION = """
OUTPUT FORMAT
Return ONLY valid JSON that conforms exactly to this schema — no markdown, no preamble:
{
  "recommendations": [
    {
      "title": string,
      "year": integer,
      "custom_genre": string,
      "movielens_genres": [string],
      "synopsis": string,
      "why_match": string,
      "rating": float,
      "streaming": string | null,
      "source": string
    }
  ],
  "follow_up_question": string | null
}
"""
structured_agent = Agent(
    model                  = model,
    db                     = agent_db,
    knowledge              = knowledge,
    tools                  = [SerperTools(), search_tmdb, get_tmdb_movie_details],
    session_id             = 'structured-demo',
    add_history_to_context = True,
    instructions           = AGENT_SYSTEM_PROMPT + STRUCTURED_OUTPUT_INSTRUCTION
                             + "\nCurrent memory is: {chat_history}",
    output_schema          = RecommendationResponse,   # ← correct parameter
    input_schema           = MoviePreference,           # ← optional: validates input dicts
    markdown               = False,
    debug_mode             = False
)

prefs = MoviePreference(viewing_context='date', audience='adults',
                        genre_mood='spy thriller', era_preference='modern', awards_interest=True
) # type: ignore

resp = structured_agent.run(input=prefs, stream=False)

print(resp.content.model_dump_json(indent=2))



{
  "recommendations": [
    {
      "title": "Tinker Tailor Soldier Spy",
      "year": 2011,
      "custom_genre": "Global Espionage Thrillers",
      "movielens_genres": [
        "Thriller",
        "Drama",
        "Mystery"
      ],
      "synopsis": "A retired spy is pulled back into the world of espionage to uncover a Soviet mole within the British intelligence service during the Cold War.",
      "why_match": "This modern spy thriller is critically acclaimed and has earned multiple award nominations, making it an excellent choice for an adult date night seeking sophisticated and gripping espionage drama.",
      "rating": 7.1,
      "streaming": "Netflix",
      "source": "https://www.imdb.com/title/tt1340800/"
    },
    {
      "title": "Mission: Impossible - Fallout",
      "year": 2018,
      "custom_genre": "Global Espionage Thrillers",
      "movielens_genres": [
        "Action",
        "Adventure",
        "Thriller"
      ],
      "synopsis": "Ethan Hunt and his team

### V8 - TMDB REST API Integration

In [67]:
# ─────────────────────────────────────────────────────────────────────────────
# Custom @tool: TMDB REST API Integration
# ─────────────────────────────────────────────────────────────────────────────
@tool
def search_tmdb(query: str, max_results: int = 3) -> str:
    """Search The Movie Database (TMDB) for movies matching the query.
    Returns JSON with title, year, overview, rating, and tmdb_id."""
    if not TMDB_API_KEY:
        return json.dumps({'error': 'TMDB_API_KEY not set in .env file'})
    url    = 'https://api.themoviedb.org/3/search/movie'
    params = {'api_key': TMDB_API_KEY, 'query': query, 'page': 1}
    try:
        resp = requests.get(url, params=params, timeout=8)
        resp.raise_for_status()
        results = []
        for m in resp.json().get('results', [])[:max_results]:
            results.append({
                'title'   : m.get('title'),
                'year'    : (m.get('release_date') or '')[:4],
                'overview': (m.get('overview') or '')[:200],
                'rating'  : m.get('vote_average'),
                'tmdb_id' : m.get('id'),
                'poster'  : f"https://image.tmdb.org/t/p/w200{m.get('poster_path','')}"
            })
        return json.dumps(results, indent=2)
    except Exception as e:
        return json.dumps({'error': str(e)})


@tool
def get_tmdb_movie_details(tmdb_id: int) -> str:
    """Fetch full details for a TMDB movie by its numeric ID.
    Returns budget, runtime, genres, top-5 cast, rating, and homepage."""
    if not TMDB_API_KEY:
        return json.dumps({'error': 'TMDB_API_KEY not set'})
    url    = f'https://api.themoviedb.org/3/movie/{tmdb_id}'
    params = {'api_key': TMDB_API_KEY, 'append_to_response': 'credits'}
    try:
        resp = requests.get(url, params=params, timeout=8)
        resp.raise_for_status()
        m    = resp.json()
        cast = [c['name'] for c in m.get('credits', {}).get('cast', [])[:5]]
        return json.dumps({
            'title'   : m.get('title'),
            'year'    : (m.get('release_date') or '')[:4],
            'genres'  : [g['name'] for g in m.get('genres', [])],
            'runtime' : m.get('runtime'),
            'rating'  : m.get('vote_average'),
            'budget'  : m.get('budget'),
            'revenue' : m.get('revenue'),
            'cast'    : cast,
            'tagline' : m.get('tagline'),
            'homepage': m.get('homepage')
        }, indent=2)
    except Exception as e:
        return json.dumps({'error': str(e)})


print('TMDB tools defined: search_tmdb, get_tmdb_movie_details')

TMDB tools defined: search_tmdb, get_tmdb_movie_details


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TMDB Agent
# ─────────────────────────────────────────────────────────────────────────────
PROMPT_AGENTV8 = """TMDB TOOL INTEGRATION
You have access to two tools that query TMDB:
1. search_tmdb(query: str, max_results: int = 3) -> str
   - Use this to find movies matching user preferences.
   - Input: a search query based on user preferences (e.g. romantic samurai movies)
   - Output: JSON list of matching movies with title, year, overview, rating, and tmdb_id.
2. get_tmdb_movie_details(tmdb_id: int) -> str
   - Use this to get detailed info on a specific movie by its TMDB ID.
   - Input: tmdb_id from search results
   - Output: JSON with budget, runtime, genres, top-5 cast, rating, and homepage.
Use these tools to enhance your movie recommendations with up-to-date info from TMDB.
Given a query, first use search_tmdb to find relevant movies, then use TMDB_ID for the movie chosen by the user to get additional details.  Always cite TMDB as your source for movie info.
   """

agent_v8 = Agent(
    model                  = model,
    db                     = agent_db,
    tools                  = [
        search_tmdb,                             #
        get_tmdb_movie_details,                  
    ],
    session_id             = "agent_v8-demo",
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV8 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)

print('✅  TMDB Agent created (added TMDB tools)')

✅  TMDB Agent created (added TMDB tools)


In [71]:
# Test  Agent V8
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v8.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v8.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you find a couple of Pink Panther movies"

Your input:  Can you find a couple of Pink Panther movies

Agent:
Happy to help with that! To find the best Pink Panther movies for you, could you tell me more about your viewing context? For example, are you watching alone, with family, or friends? And what kind of genre mood do you prefer—comedy, mystery, or something else?

Session Memory:
- Can you find a couple of Pink Panther movies
- Happy to help with that! To find the best Pink Panther movies for you, could you tell me more about your viewing context? For example, are you watching alone, with family, or friends? And what kind of genre mood do you prefer—comedy, mystery, or something else?

--------------------------------------------------

Your input:  watching alone to get me into a lighter mood

Agent:
Great! Since you want to watch alone and are aiming for a lighter mood, I assume you might enjoy the classic comedic and whimsical tone that Pink Panther movies are known for. 

Do you prefer the original classic Pink Panther

In [68]:
# ─────────────────────────────────────────────────────────────────────────────
# Full Agent: Web Search + RAG + TAG Knowledge
# ─────────────────────────────────────────────────────────────────────────────
agent = Agent(
    model                  = model,
    db                     = agent_db,
    knowledge              = knowledge,          # RAG (ABT + Oscars PDF)
    tools                  = [
        SerperTools(),                           # Pattern 1 – built-in
        search_tmdb,                             # Pattern 2 – @tool  (REST)
        get_tmdb_movie_details,                  # Pattern 2 – @tool  (REST)
    ],
    session_id             = SESSION_ID,
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV5 + "\nCurrent memory is: {chat_history}",
    markdown               = True,
    debug_mode             = False
)

print('✅  Full agent assembled: Web Search + RAG + TAG + TMDB')

✅  Full agent assembled: Web Search + RAG + TAG + TMDB


### V9 - Orchestration Decision Tree 

In [80]:
# ─────────────────────────────────────────────────────────────────────────────
# Explicit Orchestration embedded in system prompt
# Agent follows this step-by-step decision tree on every user turn
# ─────────────────────────────────────────────────────────────────────────────
ORCHESTRATION_STEPS = """
ORCHESTRATION WORKFLOW — follow exactly:

STEP 1 — CHECK MEMORY
  Read {chat_history}.  Count how many preference dimensions you already have.
  • < 2 dimensions collected → go to STEP 2
  • ≥ 2 dimensions collected → go to STEP 3

STEP 2 — COLLECT PREFERENCE
  Ask the user ONE clarifying question about the NEXT uncollected dimension
  (order: context → audience → genre → era → language → awards).
  STOP. Wait for the user's answer.

STEP 3 — CLASSIFY GENRE
  Call classify_genre(query=<combined user preferences>).
  Store the result as {custom_genre}.

STEP 4 — RETRIEVE FROM KNOWLEDGE BASE
  Search the RAG knowledge base for movies matching {custom_genre}.

STEP 5 — AUGMENT WITH WEB SEARCH
  Call SerperTools to confirm current streaming availability and any recent
  award news for the candidate movies.

STEP 6 — COMPOSE STRUCTURED RESPONSE
  Return the RecommendationResponse JSON with source citations.
"""

agent_v9 = Agent(
    model                  = model,
    db                     = agent_db,
    knowledge              = knowledge,
    tools                  = [SerperTools(), search_tmdb],
    session_id             = 'orchestrated-demo',
    add_history_to_context = True,
    instructions           = PROMPT_AGENTV5 + PROMPT_AGENTV6 + PROMPT_AGENTV8 + ORCHESTRATION_STEPS
                             + "\nCurrent memory is: {chat_history}",
    markdown               = True,
)

print('✅  Orchestrated agent ready — follows STEP 1–6 decision tree')

✅  Orchestrated agent ready — follows STEP 1–6 decision tree


In [82]:
# Test  Agent V9
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent_v9.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent_v9.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night"

Your input:  Can you recommend a movie for a date night

Agent:
Great! Are you planning to watch the movie just with your date, or will there be other people joining you?

Session Memory:
- Can you recommend a movie for a date night
- Great! Are you planning to watch the movie just with your date, or will there be other people joining you?

--------------------------------------------------

Your input:  Just two people on a date

Agent:
Thanks! And for your date night, do you and your date prefer a particular type of movie genre or mood? For example, are you in the mood for romance, comedy, thriller, or something else?

Session Memory:
- Can you recommend a movie for a date night
- Great! Are you planning to watch the movie just with your date, or will there be other people joining you?
- Just two people on a date
- Thanks! And for your date night, do you and your date prefer a particular type of movie genre or mood? For example, are you in the mood for romance, comedy, thriller, or s

---
## §5 — Integration: REST API, MCP Server, A2A
**Slides 42–54**

### 5A — REST API Integration: TMDB (Slide 50)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 50 — TMDB REST API raw call (demonstrates the JSON structure returned)
# ─────────────────────────────────────────────────────────────────────────────
def tmdb_search_raw(query: str, api_key: str = TMDB_API_KEY) -> list:
    """Direct TMDB REST call — no agent wrapper."""
    if not api_key:
        return [{'error': 'Set TMDB_API_KEY in your .env file'}]
    url    = 'https://api.themoviedb.org/3/search/movie'
    params = {'api_key': api_key, 'query': query}
    resp   = requests.get(url, params=params, timeout=8)
    data   = resp.json()
    return [{
        'id'           : m['id'],
        'title'        : m['title'],
        'year'         : m.get('release_date', '')[:4],
        'vote_average' : m.get('vote_average'),
        'poster_url'   : f"https://image.tmdb.org/t/p/w200{m.get('poster_path','')}"
    } for m in data.get('results', [])[:3]]


movies = tmdb_search_raw('Dune')
print(json.dumps(movies, indent=2))

### 5B — MCP Server Integration (Slide 44–46 & 51)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 44–46 & 51 — MCP Server Integration Pattern
#
# MCP (Model Context Protocol) wraps external systems behind a standardised
# JSON-RPC interface so ANY AI application can call them.
#
# Architecture:
#   AI Application (Claude / Custom)
#       ↓
#   MCP Client
#       ↓  (JSON-RPC over stdio / HTTP)
#   MCP Server  (e.g. IMDB MCP, TMDB MCP, Postgres MCP)
#       ↓
#   External System (Database, API, File System)
#
# Key advantages over raw REST:
#   ✓ Stateful — server maintains session context across calls
#   ✓ Discoverable — agent lists available tools dynamically from server
#   ✓ Secure — auth & audit handled centrally by the MCP server
#   ✓ Composable — same server serves Claude, ChatGPT, and custom agents
# ─────────────────────────────────────────────────────────────────────────────

MCP_SERVER_URL = os.getenv('MCP_SERVER_URL', 'http://localhost:8080/mcp')

# ── Agno MCP integration (uncomment when MCP server is running) ───────────────
# from agno.tools.mcp import MCPTools
#
# mcp_tool = MCPTools(server_url=MCP_SERVER_URL)
#
# imdb_mcp_agent = Agent(
#     model        = model,
#     tools        = [mcp_tool],         # replaces direct API calls
#     instructions = (
#         'You are a movie expert. Use the MCP server to look up IMDB data. '
#         'Call get_movie_details for full info. Call search_imdb for discovery.'
#     ),
#     markdown = True,
# )

print('MCP integration pattern defined.')
print(f'MCP Server URL: {MCP_SERVER_URL}')
print('Uncomment the MCPTools block above when your MCP server is running.')

### 5C — Agent-to-Agent (A2A) Multi-Agent Team Pattern (Slide 32 & 42)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 32 — Multi-Agent Team: Hierarchical Delegation
#
# Movie Team Orchestrator
#   ├── Oscar Analyst   — RAG over Oscars PDF, awards history
#   ├── Genre Expert    — TAG classification + TMDB discovery
#   └── Current News    — SerperTools web search
# ─────────────────────────────────────────────────────────────────────────────

# ── Sub-agent 1: Oscar Analyst ────────────────────────────────────────────────
oscar_analyst = Agent(
    name         = 'OscarAnalyst',
    model        = model,
    knowledge    = knowledge,
    instructions = (
        'You are an expert on Oscar / Academy Award history. '
        'Use the knowledge base to answer questions about nominees, winners, '
        'and historical trends. Always cite the document and year.'
    ),
    markdown     = True,
)

# ── Sub-agent 2: Genre Expert ─────────────────────────────────────────────────
genre_expert = Agent(
    name         = 'GenreExpert',
    model        = model,
    tools        = [classify_genre, search_tmdb],
    instructions = (
        'You are a movie genre expert. '
        'Use classify_genre to map queries to custom genres, '
        'then search_tmdb to find matching movies. '
        'Prefer movies with vote_average ≥ 7.0.'
    ),
    markdown     = True,
)

# ── Sub-agent 3: Current News ─────────────────────────────────────────────────
current_news_agent = Agent(
    name         = 'CurrentNews',
    model        = model,
    tools        = [SerperTools()],
    instructions = (
        'You are a movie news specialist. '
        'Search the web for current box office trends, streaming availability, '
        'and recent releases. Always include source URLs.'
    ),
    markdown     = True,
)

# ── Orchestrator / Team ────────────────────────────────────────────────────────
movie_team = Agent(
    name         = 'MovieTeamOrchestrator',
    model        = model,
    team         = [oscar_analyst, genre_expert, current_news_agent],
    instructions = [
        'You are the Movie Team Orchestrator. Route each user query to the best sub-agent.',
        'Route Oscar / award questions to OscarAnalyst.',
        'Route genre classification and movie discovery to GenreExpert.',
        'Route questions about streaming, box office, and news to CurrentNews.',
        'Synthesise sub-agent responses into one coherent answer.',
    ],
    markdown     = True,
)

print('Multi-agent team assembled:')
print('  Orchestrator : MovieTeamOrchestrator')
print('  Sub-agents   : OscarAnalyst | GenreExpert | CurrentNews')

In [ ]:
# ── Demo: route a query through the team ──────────────────────────────────────
team_query = 'What won Best Picture at the 2026 Oscars and can I stream it tonight?'
print(f'Query: {team_query}\n')
team_response = movie_team.run(team_query, stream=False)
print(team_response.content)

---
## §6 — Advanced Topics: Debugging and Context Engineering
**Slides 55–58** · debug_mode, context window control, session memory inspection, caching

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 55 — debug_mode=True: shows every tool call with inputs/outputs
# ─────────────────────────────────────────────────────────────────────────────
debug_agent = Agent(
    model        = model,
    knowledge    = knowledge,
    tools        = [SerperTools(), classify_genre],
    instructions = AGENT_SYSTEM_PROMPT,
    debug_mode   = True,    # ← prints all tool call traces to stdout
    markdown     = True,
)

print('Running with debug_mode=True — watch the tool traces below:')
debug_resp = debug_agent.run('What are some top-rated horror films from 2024?', stream=False)
print('\n── Final answer ──')
print(debug_resp.content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 56 — Context Engineering: control what enters the context window
# ─────────────────────────────────────────────────────────────────────────────

# Lever 1 — limit RAG document retrieval
knowledge_limited = Knowledge(
    vector_db   = vector_db,
    max_results = 3,    # ← fewer docs = smaller context; default is 10
)

# Lever 2 — toggle chat history injection
agent_no_memory = Agent(
    model                  = model,
    knowledge              = knowledge_limited,
    tools                  = [SerperTools()],
    add_history_to_context = False,   # ← no chat history injected
    instructions           = AGENT_SYSTEM_PROMPT,
    markdown               = True,
)

# Lever 3 — session isolation (one per user)
agent_user_42 = Agent(
    model                  = model,
    db                     = agent_db,
    knowledge              = knowledge,
    tools                  = [SerperTools(), classify_genre],
    session_id             = 'user-42',   # ← isolates memory per user
    add_history_to_context = True,
    instructions           = AGENT_SYSTEM_PROMPT + "\nCurrent memory is: {chat_history}",
    markdown               = True,
)

print('Context engineering examples defined')
print('  knowledge_limited : max_results=3 (reduced context)')
print('  agent_no_memory   : add_history_to_context=False')
print('  agent_user_42     : session_id=user-42 (isolated memory)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 56 — Inspect stored session memory
# ─────────────────────────────────────────────────────────────────────────────
chat_history = agent.get_chat_history()
print(f'Session memory: {len(chat_history)} messages stored')
for i, msg in enumerate(chat_history[:4]):
    print(f'  [{i+1}] {msg.role}: {str(msg.content)[:120]}...')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 57 — Enterprise Pattern: in-session cache + timeout circuit breaker
# ─────────────────────────────────────────────────────────────────────────────
import concurrent.futures, time

_genre_cache: dict = {}

@tool
def classify_genre_cached(query: str) -> str:
    """Genre classifier with session-level cache to avoid redundant calls."""
    key = query.strip().lower()
    if key in _genre_cache:
        print(f'  [CACHE HIT] {key}')
        return _genre_cache[key]
    result = classify_genre.entrypoint(query=query)
    _genre_cache[key] = result
    return result


@tool
def search_tmdb_safe(query: str, timeout_secs: int = 3) -> str:
    """TMDB search with 3-second circuit breaker."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(search_tmdb.entrypoint, query=query)
        try:
            return future.result(timeout=timeout_secs)
        except concurrent.futures.TimeoutError:
            return json.dumps({'error': f'TMDB timed out after {timeout_secs}s'})
        except Exception as e:
            return json.dumps({'error': str(e)})


# Test cache
r1 = classify_genre_cached.entrypoint(query='spy thriller action')
r2 = classify_genre_cached.entrypoint(query='spy thriller action')   # hits cache
print(f'Result: "{r1}" (second call used cache)')

---
## §7 — Batch Test Queries (non-interactive)
**Slide 31** · Test the full agent against representative queries

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 31 — Batch test (non-interactive) — demonstrates all capabilities
# ─────────────────────────────────────────────────────────────────────────────
test_queries = [
    'Recommend a romantic comedy for date night.',
    'Who won Best Picture at the 2026 Oscars?',
    'What are the latest Bollywood releases in 2025?',
    'Is Oppenheimer available on Netflix?',
    'Summarize the chat so far',
]

for q in test_queries:
    print(f'\nQuery: {q}')
    response = agent.run(q, stream=False)
    print('Bot:', response.content)
    print('-' * 80)

---
## §8 — Full Interactive Chatbot (Flipped Interaction)
**Slide 31** · Real conversational loop with session memory display

Type your movie question. Type `exit` to stop.

**Suggested starter queries:**
- *"Can you recommend a movie for a date night? I love romantic movies and my spouse likes Samurai fiction"*
- *"How about a movie like the series Shogun"*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SLIDE 31 — Full interactive chatbot with session memory
# ─────────────────────────────────────────────────────────────────────────────
while True:
    user_input = input('You: ')
    print('Your input: ', user_input)

    if user_input.lower().strip() == 'exit':
        print('Goodbye!')
        break

    run = agent.run(user_input, stream=False)

    print('\nAgent:')
    print(run.content)

    # Display session memory after each turn
    chat_memory = agent.get_chat_history()
    print('\nSession Memory:')
    for m in chat_memory:
        print('-', m.content)

    print('\n' + '-' * 50 + '\n')